In [1]:
import os

try:
    # 1. NẾU CHẠY TRÊN GOOGLE COLAB
    from google.colab import drive
    drive.mount('/content/drive')
    print("Đang chạy trên Google Colab.")

    # Cấu hình đường dẫn tuyệt đối trên Google Drive
    DATA_PATH = "/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_features_scaled.csv"
    MODEL_DIR = "/content/drive/MyDrive/Explainable-AML-GAT-ResNet/models/"

except ModuleNotFoundError:
    # 2. NẾU CHẠY TRÊN LOCAL (VS CODE / JUPYTER LOCAL)
    print("Đang chạy trên môi trường Local (VS Code). Bỏ qua bước mount Drive.")

    # Cấu hình đường dẫn tương đối (tương thích với cấu trúc thư mục GitHub)
    # Giả sử file notebook đang đặt trong thư mục notebooks/ hoặc models/
    DATA_PATH = "../data/elliptic_features_scaled.csv"
    MODEL_DIR = "../models/"

# Đảm bảo thư mục lưu model luôn tồn tại
os.makedirs(MODEL_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
# Import initial library
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import joblib
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Load data
DATA_PATH = "/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_features_scaled.csv"

print("Loading data...")
df = pd.read_csv(DATA_PATH)

labeled_df = df[df['y'] != -1]
print(f"Total labeled transactions: {len(labeled_df)}")

feature_cols = [col for col in df.columns if col not in ['node_idx', 'txId', 'time_step_raw', 'y']]

Loading data...
Total labeled transactions: 46564


In [4]:
# Split Train/Test
train_df = labeled_df[labeled_df['time_step_raw'] <= 34]
test_df = labeled_df[labeled_df['time_step_raw'] >= 35]

X_train = train_df[feature_cols]
y_train = train_df['y']

X_test = test_df[feature_cols]
y_test = test_df['y']

print(f"Training set: {X_train.shape[0]} samples (Time steps 1-34)")
print(f"Testing set: {X_test.shape[0]} samples (Time steps 35-49)")

Training set: 29894 samples (Time steps 1-34)
Testing set: 16670 samples (Time steps 35-49)


In [5]:
# Train Logistic Regression
print("Training Logistic Regression model...")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

lr_y_pred = lr_model.predict(X_test)

print("\n--- Logistic Regression Results ---")
print(f"Precision (Illicit): {precision_score(y_test, lr_y_pred):.3f}")
print(f"Recall (Illicit): {recall_score(y_test, lr_y_pred):.3f}")
print(f"F1 Score (Illicit): {f1_score(y_test, lr_y_pred):.3f}")
print("\nClassification Report:\n", classification_report(y_test, lr_y_pred, target_names=['Licit (0)', 'Illicit (1)']))

Training Logistic Regression model...

--- Logistic Regression Results ---
Precision (Illicit): 0.318
Recall (Illicit): 0.712
F1 Score (Illicit): 0.440

Classification Report:
               precision    recall  f1-score   support

   Licit (0)       0.98      0.89      0.93     15587
 Illicit (1)       0.32      0.71      0.44      1083

    accuracy                           0.88     16670
   macro avg       0.65      0.80      0.69     16670
weighted avg       0.94      0.88      0.90     16670



In [6]:
# Train Random Forest
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=50, max_features=50, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_y_pred = rf_model.predict(X_test)

print("\n--- Random Forest Results ---")
print(f"Precision (Illicit): {precision_score(y_test, rf_y_pred):.3f}")
print(f"Recall (Illicit): {recall_score(y_test, rf_y_pred):.3f}")
print(f"F1 Score (Illicit): {f1_score(y_test, rf_y_pred):.3f}")
print("\nClassification Report:\n", classification_report(y_test, rf_y_pred, target_names=['Licit (0)', 'Illicit (1)']))

Training Random Forest model...

--- Random Forest Results ---
Precision (Illicit): 0.911
Recall (Illicit): 0.725
F1 Score (Illicit): 0.807

Classification Report:
               precision    recall  f1-score   support

   Licit (0)       0.98      1.00      0.99     15587
 Illicit (1)       0.91      0.72      0.81      1083

    accuracy                           0.98     16670
   macro avg       0.95      0.86      0.90     16670
weighted avg       0.98      0.98      0.98     16670



In [7]:
# Save Model on Drive
import os
MODEL_DIR = "/content/drive/MyDrive/Explainable-AML-GAT-ResNet/models/"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(lr_model, os.path.join(MODEL_DIR, 'logistic_regression_model.joblib'))
joblib.dump(rf_model, os.path.join(MODEL_DIR, 'random_forest_model.joblib'))

print(f"Models successfully saved to {MODEL_DIR}")

Models successfully saved to /content/drive/MyDrive/Explainable-AML-GAT-ResNet/models/
